In [1]:
%pip install -q dotenv llama_stack_client==0.4.2

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from dotenv import load_dotenv

from llama_stack_client import LlamaStackClient

In [3]:
load_dotenv()
base_url = os.getenv("REMOTE_BASE_URL", "http://localhost:8321")
tavily_search_api_key = os.getenv("TAVILY_SEARCH_API_KEY")
wolfram_alpha_api_key = os.getenv("WOLFRAM_ALPHA_API_KEY")
client = LlamaStackClient(
    base_url=base_url,
    provider_data={"tavily_search_api_key": tavily_search_api_key, "wolfram_alpha_api_key": wolfram_alpha_api_key},
)

In [4]:
MODEL = "vllm/qwen3-8b"
INSTRUCTIONS = """You are a helpful websearch assistant. When you are asked to search the latest you must use a tool. 
Whenever a tool is called, be sure return the response in a friendly and helpful tone."""

TOOLS = [
    {"type": "web_search"},
    {
        "type": "function",
        "name": "wolfram_alpha",
        "description": "Query WolframAlpha for computational knowledge and math calculations",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "The query to compute"}
            },
            "required": ["query"],
        },
    },
]

In [5]:
def execute_tool_call(response):
    """Extract function tool calls from response, execute via tool_runtime, and continue."""
    function_calls = []
    for item in response.output:
        item_type = getattr(item, "type", None)
        if item_type == "function_call":
            function_calls.append(item)

    if not function_calls:
        return None

    # Build input with function call outputs
    tool_inputs = []
    for fc in function_calls:
        import json
        args = json.loads(fc.arguments) if isinstance(fc.arguments, str) else fc.arguments
        print(f"\n🔧 Calling {fc.name}({args})")

        # Execute via llama-stack tool_runtime
        result = client.tool_runtime.invoke_tool(
            tool_name=fc.name,
            kwargs=args,
        )
        output = result.content if hasattr(result, "content") else str(result)
        print(f"📋 Result: {str(output)[:300]}")

        tool_inputs.append({
            "type": "function_call_output",
            "call_id": fc.call_id,
            "output": str(output),
        })

    # Continue conversation with tool results
    return client.responses.create(
        model=MODEL,
        input=tool_inputs,
        instructions=INSTRUCTIONS,
        tools=TOOLS,
        stream=True,
        previous_response_id=response.id,
    )

In [6]:

def stream_with_tools(stream):
    """Stream responses API events, handling server-side and client-side tool calls."""
    final_response = None

    for event in stream:
        event_type = getattr(event, "type", None)

        # Text streaming
        if event_type == "response.output_text.delta":
            print(getattr(event, "delta", ""), end="", flush=True)

        # Refusal
        elif event_type == "response.refusal.delta":
            print(getattr(event, "delta", ""), end="", flush=True)

        # Web search status
        elif event_type == "response.output_item.added":
            item = getattr(event, "item", None)
            if item and getattr(item, "type", None) == "web_search_call":
                print(f"\n🔍 Web search")
        elif event_type == "response.web_search_call.searching":
            print("  searching...")
        elif event_type == "response.web_search_call.completed":
            print("  search completed")

        # Capture completed response for function call handling
        elif event_type == "response.completed":
            final_response = event.response

    print()

    # If there are function (client-side) tool calls, execute and continue
    if final_response:
        next_stream = execute_tool_call(final_response)
        if next_stream:
            stream_with_tools(next_stream)  # recurse for multi-turn

In [7]:
question = "Who is the current F1 World Champion?"

In [8]:
stream = client.responses.create(
    model=MODEL,
    input=question,
    instructions=INSTRUCTIONS,
    tools=TOOLS,
    stream=True,
    max_infer_iters=3,
)

stream_with_tools(stream)

INFO:httpx:HTTP Request: POST http://llamastack-distribution-service:8321/v1/responses "HTTP/1.1 200 OK"


<think>
Okay, the user is asking who the current F1 World Champion is. I need to find the most up-to-date information. Let me think about the tools I have. There's the web_search function which can search the web for recent data. Since F1 champions can change each year, I should use that to get the latest news. I don't think Wolfram Alpha would have this information, so I'll stick with web_search. The query should be straightforward: "current F1 World Champion". Let me make sure to use the correct parameters and structure the tool call properly.
</think>


🔍 Web search
  searching...
  search completed
<think>
Okay, the user asked who the current F1 World Champion is. I need to check the latest information. Let me look at the tool responses.

The first result from the web search says Lando Norris is the current champion, winning in the 2025 season. The FIA hasn't officially declared it yet. Another Wikipedia link also mentions Lando Norris as the 2025 champion. There are news articles 

In [9]:
question = "(7 * 12 ^ 10) / 321 ? and How many calories are there in a pound of strawberries?"

In [10]:
stream = client.responses.create(
    model=MODEL,
    input=question,
    instructions=INSTRUCTIONS,
    tools=TOOLS,
    stream=True,
    max_infer_iters=3,
)

stream_with_tools(stream)

INFO:httpx:HTTP Request: POST http://llamastack-distribution-service:8321/v1/responses "HTTP/1.1 200 OK"


<think>
Okay, let's tackle the user's questions. First, they want to know the result of (7 * 12^10) / 321. That looks like a math problem. I should use the Wolfram Alpha function for that. Let me check if I need to compute 12^10 first. Yeah, exponentiation first, then multiply by 7, then divide by 321. Wolfram Alpha can handle that.

Next, the user is asking how many calories are in a pound of strawberries. That's a nutritional question. I should use the web search tool here because Wolfram Alpha might not have the most up-to-date or specific food data. But maybe Wolfram Alpha can still provide a reliable answer. Wait, the user might be looking for a general estimate. Let me check both. First, I'll call Wolfram Alpha for the math part and then use web search for the calorie information to ensure accuracy.
</think>



🔍 Web search
  searching...
  search completed


🔧 Calling wolfram_alpha({'query': '(7 * 12^10) / 321'})


/tmp/ipykernel_450/2263806093.py:20: DeprecationWarning: deprecated
  result = client.tool_runtime.invoke_tool(
INFO:httpx:HTTP Request: POST http://llamastack-distribution-service:8321/v1/tool-runtime/invoke "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://llamastack-distribution-service:8321/v1/responses "HTTP/1.1 200 OK"


📋 Result: {"queryresult": {"success": true, "kernelId": "797", "processId": 1449116, "inputstring": "(7 * 12^10) / 321", "sbsallowed": false, "parentId": "5ab75719-3a0e-4b4a-b27e-3d309530ec67", "requestId": "5ab75719-3a0e-4b4a-b27e-3d309530ec67", "pods": [{"title": "Input", "subpods": [{"title": "", "plaintex
<think>
Okay, let's tackle the user's questions. First, they asked for the value of (7 * 12^10) / 321. I used Wolfram Alpha to compute this. The exact result is 144473849856/107, which as a decimal is approximately 1.35022 x 10^9. The mixed fraction is 1350222895 91/107. That's a pretty big number!

Next, the user wanted to know how many calories are in a pound of strawberries. The web search results showed some variation. The first source mentions 140 calories, another says 150, and a third notes 145. There's also a mention of 151 calories. The FatSecret link talks about 1 calorie in a quarter pound, so a full pound would be 4 calories, but that seems way off. Maybe that's a typo